In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot

In [2]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.colors import LogNorm
# Aesthetics:
fs = 14    # fontsize

In [3]:
# Check out the structure
path = "/home/pira/Documenti/PoD/LCP/LCP_B/ALICE/AO2DtreeMC.root"
file = uproot.open(path)
file.classnames()

{'DF_2303121152302944;1': 'TDirectory',
 'DF_2303121152302944/O2mccollision;1': 'TTree',
 'DF_2303121152302944/O2collision_001;1': 'TTree',
 'DF_2303121152302944/O2filtertrack;1': 'TTree',
 'DF_2303121152302944/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302944/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302944/O2genparticles;1': 'TTree',
 'DF_2303121152302976;1': 'TDirectory',
 'DF_2303121152302976/O2mccollision;1': 'TTree',
 'DF_2303121152302976/O2collision_001;1': 'TTree',
 'DF_2303121152302976/O2filtertrack;1': 'TTree',
 'DF_2303121152302976/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302976/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302976/O2genparticles;1': 'TTree',
 'DF_2303121152303008;1': 'TDirectory',
 'DF_2303121152303008/O2mccollision;1': 'TTree',
 'DF_2303121152303008/O2collision_001;1': 'TTree',
 'DF_2303121152303008/O2filtertrack;1': 'TTree',
 'DF_2303121152303008/O2filtertrackextr;1': 'TTree',
 'DF_2303121152303008/O2filtertrackmc;1': 'TTree',
 'DF_2303121152303008

In [4]:
# Function to filter out the "good" couples of tracks
def check_group(group):
    part_pdg = set(group['fPdgCode'])
    return -321 in part_pdg and 211 in part_pdg and len(group) == 2 


In [23]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

#good_gen_part = pd.DataFrame()
#tot_gen_part = pd.DataFrame()
n_D_0 = 0

for directory in file_ID:

    tracks_df = file[directory + "/O2filtertrack"].arrays(["fIndexCollisions", ]
                                                          ,library="pd")
    tracks_mc_df = file[directory + "/O2filtertrackmc"].arrays(["fPdgCode","fMainMotherOrigIndex","fMainHfMotherPdgCode","fMainMotherNfinalStateDaught","fMainBeautyAncestorPdgCode"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_mc_df, 
                         how='inner', left_index=True, right_index=True)

    tracks_extra = file[directory + "/O2filtertrackextr"].arrays(["fEta"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_extra, 
                         how='inner', left_index=True, right_index=True)

    collision_fPosZ = file[directory + "/O2collision_001"].arrays( ["fPosZ"], library="pd")
    tracks_df = pd.merge(left=tracks_df, right=collision_fPosZ, how='inner', left_on='fIndexCollisions', right_index=True)

    # fPosZ cut
    tracks_df = tracks_df[tracks_df["fPosZ"].abs() < 10]

    # Select the right particles (fPdgCode)
    tracks_df = tracks_df[tracks_df["fPdgCode"].isin([-321,211])]
    
    # Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
    tracks_df = tracks_df[(tracks_df["fMainHfMotherPdgCode"] == 421) ]
                       # & (tracks_df['fMainMotherNfinalStateDaught'] >= 2)]

    # Pseudorapidity cut
    tracks_df = tracks_df[tracks_df["fEta"].abs() < 0.8]

    # Check on the fMainBeautyAncestor = 0
    tracks_df = tracks_df[tracks_df["fMainBeautyAncestorPdgCode"] == 0]

    
    # fMainMotherOrigIndex is the same for the grouped tracks
    tracks_df = tracks_df.groupby('fMainMotherOrigIndex')
   

    

    #defined pT interval in which you can/want to “make the measurement”
    # For now no selection on pT...TO DO LATER
    
    # Applica il filtro sui gruppi
    D_0_decays = tracks_df.filter(check_group)


    # Contiamo le D_0 e sommiamole per avere un totale
    n_D_0 += len(D_0_decays)/2

    print("In this file there are", len(D_0_decays)/2, "D_0 decays")
    

In this file there are 2.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 9.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 3.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 7.0 D_0 decays
In this file there are 2.0 D_0 decays
In this file there are 10.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 7.0 D_0 decays
In this file there are 7.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 5.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 3.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 4.0 D_0 decays
In this fil

In [24]:
n_D_0

1378.0

In [18]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

good_gen_part = pd.DataFrame()
tot_gen_part = pd.DataFrame()

gen_D_0 = 0



for directory in file_ID:

     # "JOIN"
    generated_part = file[directory + "/O2genparticles"].arrays(library="pd")
    mc_collision_df = file[directory + "/O2mccollision"].arrays(library="pd")
    generated_part = pd.merge(left=generated_part, right=mc_collision_df, how='inner', left_on='fIndexMcCollisions', right_index=True)

    
    # fPosZ cut
    generated_part = generated_part[generated_part["fPosZ"].abs() < 10]

    # Pseudorapidity cut
    generated_part = generated_part[generated_part["fMainMotherY"].abs() < 0.8]
    
    # Check on the fMainBeautyAncestor = 0
    generated_part = generated_part[generated_part["fMainBeautyAncestorPdgCode"]==0]

    generated_part = generated_part[generated_part["fPdgCode"]==421]

    n_gen_D_0 = len(generated_part)
    print("In this file there are", n_gen_D_0, "D_0 generated")

    gen_D_0 += n_gen_D_0 
    
    #defined pT interval in which you can/want to “make the measurement”
    # For now no selection on pT...TO DO LATER 
    
    #results
    #good_gen_part = pd.concat([good_gen_part, generated_part], ignore_index=True) 
    #tot_gen_part = pd.concat([tot_gen_part, generated_part], ignore_index=True) 
    

#print("The generated particles before check the Z coordinate are:", len(tot_gen_part))
#print("After the cut on Z_collision < 10 cm we have ", len(good_gen_part), "particles")

In this file there are 52 D_0 generated
In this file there are 28 D_0 generated
In this file there are 32 D_0 generated
In this file there are 57 D_0 generated
In this file there are 22 D_0 generated
In this file there are 25 D_0 generated
In this file there are 23 D_0 generated
In this file there are 48 D_0 generated
In this file there are 22 D_0 generated
In this file there are 21 D_0 generated
In this file there are 23 D_0 generated
In this file there are 23 D_0 generated
In this file there are 33 D_0 generated
In this file there are 49 D_0 generated
In this file there are 20 D_0 generated
In this file there are 27 D_0 generated
In this file there are 31 D_0 generated
In this file there are 30 D_0 generated
In this file there are 30 D_0 generated
In this file there are 27 D_0 generated
In this file there are 26 D_0 generated
In this file there are 23 D_0 generated
In this file there are 20 D_0 generated
In this file there are 53 D_0 generated
In this file there are 58 D_0 generated


In [19]:
gen_D_0

10962